In [ ]:
"""
Construit le tableau de machine learning : 1 ligne = 1 génome d'E. coli.
Colonnes : y (1 = résistant, 0 = sensible), mlst_group (pour GroupKFold),
puis les locus du pangénome (1 = gène présent, 0 = absent).
Version notebook : modifie AB pour changer d'antibiotique.
"""
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------- paramètres
AB = "ciprofloxacin"        # ex. "ampicillin", "cefotaxime", "piperacillin/tazobactam"
MIN_FREQ = 0.01             # retire les locus présents chez < 1 % ou > 99 % des génomes

ROOT = Path(r"C:\Users\Alexandre\Documents\Agro Rennes\Projet_ML_and_DL-main\Projet_ML_and_DL")
D = ROOT / "data" / "Ecoli_AMR_GenotypePhenotype" / "Ecoli_AMR_GenotypePhenotype"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------- 1. lecture du fichier PLINK
def read_plink_presence_absence(prefix: Path) -> pd.DataFrame:
    """Lit un .bed PLINK et renvoie un DataFrame génomes x locus (int8, 1 = présent).

    Dans ce jeu, l'allèle '2' = gène présent et l'allèle '1' = absent
    (vérifié en comparant au fichier pangenome/whole_pan_ecor_presence_absence.csv).
    """
    fam = pd.read_csv(prefix.with_suffix(".fam"), sep=r"\s+", header=None, dtype=str)
    bim = pd.read_csv(prefix.with_suffix(".bim"), sep="\t", header=None, dtype=str)
    samples, loci = fam[1].values, bim[0].values
    n, m = len(samples), len(loci)
    bytes_per_locus = (n + 3) // 4

    raw = np.fromfile(prefix.with_suffix(".bed"), dtype=np.uint8)
    if raw[:3].tolist() != [0x6C, 0x1B, 0x01]:
        raise ValueError("Fichier .bed inattendu (pas en mode SNP-major)")
    raw = raw[3:].reshape(m, bytes_per_locus)

    # chaque octet contient 4 génomes codés sur 2 bits :
    # 00 = homozygote allèle A1 (col. 5 du .bim), 11 = homozygote allèle A2 (col. 6)
    codes = np.empty((m, bytes_per_locus * 4), dtype=np.uint8)
    for k, shift in enumerate((0, 2, 4, 6)):
        codes[:, k::4] = (raw >> shift) & 0b11
    codes = codes[:, :n]

    a1_is_present = (bim[4].values == "2")[:, None]   # allèle A1 = '2' -> présent
    a2_is_present = (bim[5].values == "2")[:, None]
    present = np.where(codes == 0, a1_is_present, np.where(codes == 3, a2_is_present, False))
    return pd.DataFrame(present.T.astype(np.int8), index=samples, columns=loci)


X = read_plink_presence_absence(D / "presence_absence_data" / "presence_absence_all")
print(f"Matrice présence/absence : {X.shape[0]} génomes x {X.shape[1]} locus")

# ------------------------------------------------------- 2. cible (phénotype)
ph = pd.read_csv(D / "phenotype_matrix" / "phenotype_matrix_08302024.csv",
                 usecols=["Genome.Name", AB], low_memory=False)
print(f"\n{AB} : {ph[AB].value_counts(dropna=False).to_dict()}")
ph = ph[ph[AB].isin(["Resistant", "Susceptible"])].copy()   # on écarte Intermediate et NaN
ph["y"] = (ph[AB] == "Resistant").astype(np.int8)

# ------------------------------------------ 3. passerelle Genome.Name -> SRA
sra = pd.read_csv(D / "phenotype_matrix" / "SRA_to_genome_name.csv",
                  usecols=["Genome.Name", "SRA.Accession"])
df = ph.merge(sra, on="Genome.Name", how="inner")

# ---------------------------------------------------- 4. lignée (MLST Achtman)
meta = pd.read_csv(D / "strains_metadata.csv", usecols=["Genome.Name", "MLST"])
meta["mlst_group"] = "ST" + meta["MLST"].str.extract(r"Escherichia_coli_1\.(\d+)")[0]
df = df.merge(meta[["Genome.Name", "mlst_group"]], on="Genome.Name", how="left")
df["mlst_group"] = df["mlst_group"].fillna("unknown")

# ------------------------------------------------- 5. jointure avec les gènes
df = df.drop_duplicates("SRA.Accession").set_index("SRA.Accession")
df = df[["Genome.Name", "mlst_group", "y"]].join(X, how="inner")
df.index.name = "sra"
del X   # libère la mémoire

# ------------------------------------- 6. filtre des locus peu informatifs
loci = df.columns[3:]
freq = df[loci].mean()
keep = freq[(freq >= MIN_FREQ) & (freq <= 1 - MIN_FREQ)].index
df = df[["Genome.Name", "mlst_group", "y", *keep]]

# ------------------------------------------------------------- 7. résumé
print(f"\nTableau final : {df.shape[0]} génomes x {len(keep)} locus "
      f"(sur {len(loci)} avant filtre)")
print(f"Résistants : {int(df['y'].sum())} ({df['y'].mean():.1%})")
print(f"Groupes MLST : {df['mlst_group'].nunique()} "
      f"(dont 'unknown' : {(df['mlst_group'] == 'unknown').sum()} génomes)")
assert df.index.is_unique

name = AB.replace("/", "_").replace(" ", "_")
out = OUT / f"ecoli_{name}.parquet"
df.to_parquet(out)
print(f"\nSauvegardé : {out}")

Matrice présence/absence : 7043 génomes x 32441 locus

ciprofloxacin : {'Susceptible': 4657, 'Resistant': 1291, nan: 944, 'Intermediate': 91}

Tableau final : 5935 génomes x 20731 locus (sur 32441 avant filtre)
Résistants : 1285 (21.7%)
Groupes MLST : 423 (dont 'unknown' : 406 génomes)

Sauvegardé : C:\Users\Alexandre\Documents\Agro Rennes\Projet_ML_and_DL-main\Projet_ML_and_DL\data\processed\ecoli_ciprofloxacin.parquet


In [7]:
# ------------------------------------------------- 8. export CSV du tableau final
out_csv = OUT / f"ecoli_{name}.csv"
df.to_csv(out_csv)
print(f"Exporté en CSV : {out_csv}")

Exporté en CSV : C:\Users\Alexandre\Documents\Agro Rennes\Projet_ML_and_DL-main\Projet_ML_and_DL\data\processed\ecoli_ciprofloxacin.csv
